# 03 DataFrames

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Understand what a Spark DataFrame is, create one in several ways, and inspect it.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
The DataFrame is the object you will use in almost every PySpark lesson and every real pipeline. Interviewers often ask how a DataFrame differs from an RDD and from a Pandas DataFrame.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What is a DataFrame?</b><br>
A distributed table of data organized into named columns, each with a data type (the schema).
<ul>
<li><b>Distributed:</b> rows are split into partitions spread across executors</li>
<li><b>Immutable:</b> you never change a DataFrame, each transformation returns a new one</li>
<li><b>Lazy:</b> transformations only build a plan, work starts when you call an action</li>
<li><b>Schema:</b> every column has a name and a type</li>
</ul>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Example</b><br>
A bank loads 10 million transactions into a DataFrame with columns <code>txn_id</code>, <code>account</code>, <code>amount</code> and <code>txn_date</code>. Spark splits it across executors, and every filter or aggregation you write runs on all partitions in parallel.
</div>

## DataFrame vs RDD vs Pandas DataFrame

| Name | What it is |
|---|---|
| RDD | The original low-level Spark API. A distributed collection of objects with no schema |
| Spark DataFrame | A distributed table with a schema. Spark can optimize it with Catalyst |
| Pandas DataFrame | A table on a single machine, held entirely in memory |

```
Spark DataFrame (distributed)             Pandas DataFrame (single machine)

  Partition 1  -> Executor 1                +-------------------------+
  Partition 2  -> Executor 2                |  All rows in one memory |
  Partition 3  -> Executor 3                +-------------------------+
```

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Rule of thumb</b><br>
Use DataFrames in Spark by default. RDDs are only needed for special low-level cases.
</div>

## 1. Create a DataFrame from a list of tuples

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds a DataFrame from a Python list, where each tuple is one row.<br><br>
<b>Why it matters</b><br>
It is the quickest way to create test data for learning and debugging.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 4-row table with columns <code>id</code>, <code>name</code>, <code>country</code> and <code>salary</code>.
</div>

In [0]:
# Our data: a list of tuples. Each tuple is one row.
data = [
    (1, "Amal", "India", 21000),
    (2, "Senthil", "USA", 150000),
    (3, "John", "UK", 18000),
    (4, "Vivek", "UAE", 25000),
]

# The column names, in the same order as the values in each tuple.
columns = ["id", "name", "country", "salary"]

# createDataFrame() builds a Spark DataFrame from the data and column names.
df = spark.createDataFrame(data, columns)

# show() prints the rows as a plain text table.
df.show()

## 2. Create a DataFrame from Row objects

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Builds a DataFrame using <code>Row</code>, which names each value as you write it.<br><br>
<b>Why it matters</b><br>
<code>Row</code> makes the data self-describing, because you see the column name next to each value. It is common in examples and tests.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A 2-row table with columns <code>id</code>, <code>city</code> and <code>population_millions</code>.
</div>

In [0]:
# Row lets us write each record with named fields.
from pyspark.sql import Row

# Each Row is one record. The field names become the column names.
rows = [
    Row(id=1, city="Chennai", population_millions=11.0),
    Row(id=2, city="Dubai", population_millions=3.6),
]

# No separate column list is needed, because the Rows carry their own names.
df_rows = spark.createDataFrame(rows)

# show() prints the rows as a plain text table.
df_rows.show()

## 3. Create a DataFrame from Pandas (and back)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Converts a Pandas DataFrame into a Spark DataFrame, then converts a Spark DataFrame back to Pandas.<br><br>
<b>Why it matters</b><br>
A common bridge when you prepare small data in Pandas or want to plot results.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Watch out</b><br>
<code>toPandas()</code> collects <b>all</b> rows to the driver, so only use it on small data.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A Spark DataFrame with 2 rows, then a Pandas DataFrame printed as text.
</div>

In [0]:
# Pandas is a single-machine data library that is available in Databricks.
import pandas as pd

# Build a small Pandas DataFrame from a dictionary (column name -> list of values).
pdf = pd.DataFrame({"id": [1, 2], "product": ["Laptop", "Phone"]})

# Convert the Pandas DataFrame into a Spark DataFrame.
df_from_pandas = spark.createDataFrame(pdf)
df_from_pandas.show()

# Convert a Spark DataFrame back to Pandas.
# WARNING: toPandas() pulls every row to the driver, so use it only on small data.
back_to_pandas = df_from_pandas.toPandas()
print(back_to_pandas)

## 4. Inspect the structure: schema, columns, dtypes, count

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Reads the DataFrame's schema, column names and types, then counts the rows.<br><br>
<b>Why it matters</b><br><code>printSchema()</code>, <code>columns</code> and <code>dtypes</code> only read metadata, so they are instant. <code>count()</code> is an action that scans the data. Knowing which is which is a common interview question.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A schema tree with <code>id: long</code>, <code>name: string</code>, <code>country: string</code>, <code>salary: long</code>, then the column list, the type pairs, the schema object and <code>Row count: 4</code>.
</div>

In [0]:
# printSchema() prints each column's name, data type and whether it can hold nulls.
df.printSchema()

# columns returns just the list of column names.
print("Columns:", df.columns)

# dtypes returns a list of (column name, type) pairs.
print("Types:", df.dtypes)

# schema returns the schema object, with full detail for each column.
print("Schema object:", df.schema)

# count() is an ACTION. Spark scans the data and returns the number of rows.
print("Row count:", df.count())

## 5. Look at the data: show, display, take, head, collect

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Retrieves rows in different ways.<br><br>
<b>Why it matters</b><br>
These look similar but behave differently. <code>show()</code> and <code>display()</code> are for looking, while <code>take()</code>, <code>head()</code> and <code>collect()</code> return rows to Python.
</div>
<br>
<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Common mistake</b><br>
Calling <code>collect()</code> on a large DataFrame. It brings <b>everything</b> to the driver and can crash it. Use <code>take(n)</code> or <code>show(n)</code> instead.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A text table, an interactive table, then Python <code>Row</code> objects.
</div>

In [0]:
# show(n) prints the first n rows as text. The default is 20 rows.
df.show(2)

# display() shows an interactive Databricks table (sort, filter, chart).
display(df)

# take(n) returns the first n rows to the driver as a list of Row objects.
print("take(2):", df.take(2))

# head() returns the first row (or the first n rows if you pass n).
print("head():", df.head())

# first() is the same as head() and returns one Row.
print("first():", df.first())

# collect() returns ALL rows to the driver as a list.
# Safe here because we have 4 rows. Avoid it on large data.
all_rows = df.collect()
print("collect():", all_rows)

# A Row works like a record: access a field by name or by position.
print("First name:", all_rows[0]["name"])

## 6. Summarise and limit

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Gets quick statistics with <code>describe()</code>, and creates a smaller DataFrame with <code>limit()</code>.<br><br>
<b>Why it matters</b><br>
<code>describe()</code> is a fast way to understand numeric columns. <code>limit()</code> returns a <b>new</b> DataFrame, which shows that DataFrames are immutable.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A statistics table (count, mean, stddev, min, max), then a 2-row DataFrame, then the original and limited row counts.
</div>

In [0]:
# describe() computes count, mean, stddev, min and max for the columns you give it.
df.describe("salary").show()

# limit(2) returns a NEW DataFrame with at most 2 rows.
# The original df is unchanged, because DataFrames are immutable.
small_df = df.limit(2)
small_df.show()

# Prove that df still has 4 rows.
print("Original rows:", df.count(), "| Limited rows:", small_df.count())

## 7. DataFrames are immutable

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Applies a transformation and shows that the original DataFrame is not changed.<br><br>
<b>Why it matters</b><br>
A core concept. You never modify a DataFrame. You create a new one from it, and nothing runs until an action.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The original columns without <code>bonus</code>, then the new columns with <code>bonus</code>, then a table including it.
</div>

In [0]:
# Import the helper functions module. 'F.col' refers to a column by name.
from pyspark.sql import functions as F

# withColumn() returns a NEW DataFrame with an added column.
# This is a transformation, so no data is processed yet.
df_with_bonus = df.withColumn("bonus", F.col("salary") * 0.10)

# The original df has no 'bonus' column. It was not changed.
print("Original columns:", df.columns)

# Only now, when we call show() (an action), does Spark do the work.
print("New columns:", df_with_bonus.columns)
df_with_bonus.show()

## What Spark is doing internally

```
spark.createDataFrame(data, columns)
        |
        v
  Spark creates a plan for the data      (no job yet)
        |
        v
df.withColumn(...)  -> new DataFrame     (still lazy, just extends the plan)
        |
        v
df.show() / df.count() / df.collect()    (ACTION)
        |
        v
  Spark runs a job on the executors, and returns rows to the driver
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata only</span>
<code>printSchema()</code>, <code>columns</code>, <code>dtypes</code> are instant.

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span>
<code>withColumn()</code>, <code>limit()</code> build a plan and return a new DataFrame.

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span>
<code>count()</code>, <code>show()</code>, <code>take()</code>, <code>collect()</code> trigger real work.

We study lazy evaluation and jobs in detail in <code>02_spark_internals</code>.

## Troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>toPandas()</code> is slow or the notebook crashes</b><br>
The data is too big for the driver. Filter or aggregate in Spark first.<br><br>
<b>⛔ Problem: <code>collect()</code> crashes the driver</b><br>
Same reason. Use <code>take(n)</code> or <code>show(n)</code> instead.<br><br>
<b>⛔ Problem: types show as <code>long</code> and <code>double</code></b><br>
Spark infers Python <code>int</code> as <code>long</code> and <code>float</code> as <code>double</code>. Lesson 4 covers setting types.<br><br>
<b>⛔ Problem: <code>Can not infer schema</code> error from an empty list</b><br>
Give an explicit schema (Lesson 4), because Spark can't guess from no data.<br><br>
<b>✍️ My own problems</b><br>
(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is a Spark DataFrame?</b><br>
A distributed collection of data organized into named columns with a schema, like a table. It is immutable and lazily evaluated.<br><br>

<b>🎤 2. How is a DataFrame different from an RDD?</b><br>
An RDD is a low-level collection of objects with no schema. A DataFrame has a schema, so Spark's Catalyst optimizer can plan and optimize operations. DataFrames are usually faster and easier to write.<br><br>

<b>🎤 3. How is a Spark DataFrame different from a Pandas DataFrame?</b><br>
Spark's is distributed across a cluster and lazy. Pandas runs on one machine, keeps data in memory and executes immediately.<br><br>

<b>🎤 4. What does it mean that DataFrames are immutable?</b><br>
You can't change a DataFrame in place. Each transformation returns a new DataFrame, and the original stays the same.<br><br>

<b>🎤 5. What is the difference between <code>show()</code>, <code>take()</code> and <code>collect()</code>?</b><br>
<code>show()</code> prints rows for viewing. <code>take(n)</code> returns n rows to the driver as a list. <code>collect()</code> returns all rows to the driver, which can crash it on large data.<br><br>

<b>🎤 6. Is <code>count()</code> a transformation or an action?</b><br>
An action, because it triggers a job and returns a result to the driver.<br><br>

<b>🎤 7. Is <code>printSchema()</code> an action?</b><br>
No. It only reads metadata and doesn't process data.<br><br>

<b>🎤 8. Why be careful with <code>toPandas()</code>?</b><br>
It collects the whole DataFrame into the driver's memory, so it can fail on large data.<br><br>

<b>🎤 9. Is the data held in memory once you create a DataFrame?</b><br>
Not necessarily. A DataFrame is a plan plus a description of the data. Data is processed when an action runs, unless you cache it.
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol>
<li>Create a DataFrame of 5 products with columns <code>product_id</code>, <code>name</code>, <code>category</code> and <code>price</code></li>
<li>Print its schema, column names and row count</li>
<li>Show only the first 3 rows with <code>show()</code></li>
<li>Use <code>take(2)</code> and print the result. In a comment, explain how it differs from <code>collect()</code></li>
<li>Use <code>describe("price")</code> to get the statistics</li>
<li>Create a Pandas DataFrame from a dictionary with 2 columns, convert it to Spark, then show it</li>
<li>In one sentence: why is <code>df.withColumn(...)</code> not changing <code>df</code>?</li>
</ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Summary

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul>
<li>A DataFrame is a distributed, immutable, lazily evaluated table with a schema</li>
<li>Create one from tuples, Rows, or Pandas</li>
<li><code>printSchema()</code>, <code>columns</code>, <code>dtypes</code> read metadata only</li>
<li><code>show()</code> and <code>display()</code> look at data, <code>take()</code>, <code>head()</code> and <code>collect()</code> return rows to the driver</li>
<li><code>collect()</code> and <code>toPandas()</code> bring data to the driver, so use them on small data only</li>
<li>Each transformation returns a new DataFrame</li>
</ul>
</div>
